# Phase 1 — FSQA Eval Question Generation (v2, grounded)
**FSQA** | fine-tune pipeline

Generates open-ended FSQA eval questions with reference answers, grounded in
the **actual text of each CFR section** pulled live from the eCFR API.

**What changed in v3 (2026-08-24)**
- `norm_citation()` accepts `citations` (list) as well as `citation` (str).
  v2 read only the singular key and rejected valid questions.
- eCFR fetch retries 5xx, not just 429. v2 could drop a section on a transient 503.
- The 47-call eCFR loop prints progress; v2 was silent and looked hung.
- Failed batches get one retry pass before output is written.
- Manifest records per-batch yield and flags batches returning under 60%.
- `SMOKE_TEST` defaults to False.

**What changed from v1**
- Model: `anthropic/claude-opus-5` (Fable is blocked on Bedrock BYOK by the
  `provider_data_share` requirement)
- **eCFR retrieval** — real section text goes into every prompt, so the model
  writes questions from ground truth instead of memory
- **Web search** enabled via the OpenRouter server tool
- Section headings are fetched live, not hardcoded. v1's hardcoded labels had
  the entire 21 CFR 810 series off by one.
- Question count scales to section length; a 433-character section does not
  get asked for 20 questions
- Error handling captures the HTTP response body on 4xx

Weighting: 40% FSVP (21 CFR 1 Subpart L) / 40% recall (21 CFR 7, 810)
/ 20% PCHF (21 CFR 117)

---
## Notebook setup
1. **Secret** `your-modal-secret` → must contain `OPENROUTER_API_KEY`
2. **Volume** `<MODAL_VOLUME_NAME>` → mounts at `./local-data`

Run cells in order. Cell 4 preflights. Cell 8 probes the model + tool combo
before anything expensive. `SMOKE_TEST` defaults to False in v3.

In [ ]:
import concurrent.futures
import itertools
import json
import os
import re
import time
from collections import Counter
from datetime import datetime, timezone

try:
    import requests
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "requests"], check=True)
    import requests

print("imports ok")

## Config

In [ ]:
# --- Run mode -------------------------------------------------------------
SMOKE_TEST = False         # v3 default: full 50 batches. Set True for a first
                           # smoke-test pass on your own account.

# --- Model ----------------------------------------------------------------
MODEL_SLUG = "anthropic/claude-opus-5"

# --- Web search -----------------------------------------------------------
# OpenRouter server tool. The ':online' suffix is deprecated.
# Costs extra per search; the manifest reports actual spend.
USE_WEB_SEARCH = True
WEB_SEARCH_TOOL = [{"type": "openrouter:web_search"}]

# Web search is refused until its cost is bounded. Set SEARCH_COST_BOUNDED True only after search uses, results and
# search-result context tokens are bounded and priced into the pre-dispatch reservation.
SEARCH_COST_BOUNDED = False


class SearchNotBounded(RuntimeError):
    pass


def request_enables_search(body):
    """True if a request body turns on any web search path: a search server tool, the web plugin, web_search_options,
    or an :online model suffix."""
    if not isinstance(body, dict):
        return False
    if any("search" in str(k).lower() for k in body) or ":online" in str(body.get("model", "")).lower():
        return True
    for t in body.get("tools") or []:
        if "search" in json.dumps(t, default=str).lower() or "web" in json.dumps(t, default=str).lower():
            return True
    for p in body.get("plugins") or []:
        if "web" in json.dumps(p, default=str).lower() or "search" in json.dumps(p, default=str).lower():
            return True
    return False


def require_bounded_search():
    if SEARCH_COST_BOUNDED is not True:
        raise SearchNotBounded(
            "Refusing to run: web search is billed separately from tokens, can run more than once per request, and "
            "adds search-result tokens, so the reservation does not bound the cost and search-enabled requests stay "
            "off. Set USE_WEB_SEARCH = False to run without search.")


# --- Structured output ----------------------------------------------------
# Some provider endpoints reject tools + response_format together. If the
# probe in cell 8 shows that, set this False; the prompt still demands JSON
# and the parser strips code fences.
USE_RESPONSE_FORMAT = True

# --- Provider pinning -----------------------------------------------------
# allow_fallbacks False means a failed BYOK call errors instead of silently
# falling through to OpenRouter's pool and billing the wrong account.
PROVIDER_CONFIG = {
    "order": ["amazon-bedrock"],
    "allow_fallbacks": False,
}

# --- eCFR -----------------------------------------------------------------
ECFR_BASE = "https://www.ecfr.gov/api/versioner/v1/full"
ECFR_DATE = "2026-08-01"        # snapshot date; keep fixed for reproducibility
ECFR_TITLE = "21"
ECFR_HEADERS = {"User-Agent": "labs-benchmark/1.0 (research; you@example.com)"}

# --- Paths ----------------------------------------------------------------
VOLUME_MOUNT = "./local-data"
OUTPUT_SUBDIR = "phase1_fsqa_eval"

# --- Generation -----------------------------------------------------------
MAX_WORKERS = 6
REQUEST_TIMEOUT = 600           # web search + long CFR text needs headroom
MAX_RETRIES = 4

OPENROUTER_URL = "https://openrouter.ai/api/v1/chat/completions"

API_KEY = os.environ.get("OPENROUTER_API_KEY")
if not API_KEY:
    raise RuntimeError(
        "OPENROUTER_API_KEY not found. Attach 'your-modal-secret' to this "
        "notebook and restart the kernel."
    )
if not os.path.isdir(VOLUME_MOUNT):
    raise RuntimeError(
        f"{VOLUME_MOUNT} not found. Attach the '<MODAL_VOLUME_NAME>' volume. "
        f"./local-data contains: {os.listdir('./local-data') if os.path.isdir('./local-data') else 'nothing'}"
    )

HEADERS = {
    "Authorization": f"Bearer {API_KEY}",
    "Content-Type": "application/json",
    "X-OpenRouter-Title": "labs-benchmark",
}

# --- Spend guard -----------------------------------------------------------
# Every OpenRouter call is reserved at its worst case before it is sent and settled after (guarded_post).
# A call is refused, and the run stops, if its model has no entry in PRICE, if its body has no max_tokens,
# or if its worst case would take the run past RUN_BUDGET_USD. PRICE holds each model's confirmed listed price in
# USD per million tokens (input, output). A model with no confirmed price is left out of PRICE, and the
# notebook then refuses to run before any paid call.
from datetime import datetime, timezone
import math


# --- Canonical env parsing and ledger-root rule (identical in every notebook) ---------------------------------
# RUN_BUDGET_USD: unset means the default. Any value that is set, a blank or whitespace
# value included, must parse as a finite float greater than 0, or the notebook refuses to start.
def _env_usd(name, default):
    raw = os.environ.get(name)
    if raw is None:
        return float(default)
    try:
        v = float(raw.strip())
    except ValueError:
        v = float("nan")
    if not math.isfinite(v) or v <= 0:
        raise RuntimeError(f"Refusing to start: {name} must be a finite number greater than 0 (got {raw!r}).")
    return v


RUN_BUDGET_USD = _env_usd("RUN_BUDGET_USD", 25.00)   # one cap per run; set the RUN_BUDGET_USD environment variable to change it
GEN_MAX_TOKENS = 8000      # output cap (max_tokens) for every generation call; the guard reserves against it
RUN_ID = None              # None = new run with a fresh ledger. Set a non-empty string to resume and keep its spend.
# Source: https://openrouter.ai/api/v1/models/anthropic/claude-opus-5/endpoints, retrieved 2026-11-05.
# Highest endpoint price: the Anthropic fast/priority tier. Standard global endpoints are 5 / 25.
PRICE = {"anthropic/claude-opus-5": (10.0, 50.0)}   # {"<model slug>": (input $/M, output $/M)}
missing_prices = sorted({MODEL_SLUG} - set(PRICE))
if missing_prices:
    raise RuntimeError(f"No confirmed price for {missing_prices}. Nothing was sent. Add the listed USD per million "
                       "tokens (input, output) for each model to PRICE in this config cell, then rerun.")
# None is the only signal for a new run. Any other value is a resume ID and goes through _checked_run_id below,
# which refuses an empty string, 0, False and every other non-string value. Nothing is defaulted.
if RUN_ID is None:
    RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
def _checked_run_id(value):
    """RUN_ID names one directory under SPEND_ROOT. Refuse an absolute path or any '..' segment, so the text of
    RUN_ID cannot lead os.path.join(SPEND_ROOT, RUN_ID) outside SPEND_ROOT. This is a string check only: it does
    not detect a symlink that already exists under SPEND_ROOT. Fails loudly."""
    if not isinstance(value, str) or not value.strip():
        raise RuntimeError(f"Refusing to start: RUN_ID must be a non-empty string (got {value!r}).")
    if os.path.isabs(value) or value.startswith(("/", "\\")) or os.path.splitdrive(value)[0]:
        raise RuntimeError(f"Refusing to start: RUN_ID must be a relative name, not an absolute path (got {value!r}).")
    if ".." in value.replace("\\", "/").split("/"):
        raise RuntimeError(f"Refusing to start: RUN_ID must not contain a '..' segment (got {value!r}).")
    return value
RUN_ID = _checked_run_id(RUN_ID)
RUN_DIR = os.path.join(VOLUME_MOUNT, OUTPUT_SUBDIR, RUN_ID)
os.makedirs(RUN_DIR, exist_ok=True)

# One ledger root and lock root for every notebook: SPEND_LEDGER_DIR if set, else <repo root>/spend_ledger.
# Ledger root. If SPEND_LEDGER_DIR is set at all, it is stripped: blank or whitespace-only refuses, anything else
# becomes os.path.abspath(os.path.expanduser(value)). If it is not set, the root is <repo root>/spend_ledger, where
# the repo root is the first directory, starting at the absolute working directory and walking up, for which
# os.path.exists(<dir>/.git) is true (a .git directory or a .git file). With no such directory the notebook refuses:
# there is no fallback to the working directory, because a different working directory would give a different lock.
# The lock is <root>/runner.lock and the run ledgers are <root>/<RUN_ID>/spend_ledger.jsonl.
def _ledger_root():
    raw = os.environ.get("SPEND_LEDGER_DIR")
    if raw is not None:
        value = raw.strip()
        if not value:
            raise RuntimeError("Refusing to start: SPEND_LEDGER_DIR is set but blank. Unset it to use "
                               "<repo root>/spend_ledger, or set it to a directory.")
        return os.path.abspath(os.path.expanduser(value))
    start = os.path.abspath(os.getcwd())
    d = start
    while True:
        if os.path.exists(os.path.join(d, ".git")):
            return os.path.join(d, "spend_ledger")
        parent = os.path.dirname(d)
        if parent == d:
            raise RuntimeError(f"Refusing to start: SPEND_LEDGER_DIR is not set and there is no .git above the working "
                               f"directory ({start}). Set SPEND_LEDGER_DIR to the shared ledger directory.")
        d = parent


SPEND_LEDGER_ROOT = _ledger_root()
LEDGER_DIR = os.path.join(SPEND_LEDGER_ROOT, RUN_ID)
os.makedirs(LEDGER_DIR, exist_ok=True)
LEDGER_PATH = os.path.join(LEDGER_DIR, "spend_ledger.jsonl")

# One paid runner at a time across all notebooks. A single exclusive lock file in the shared ledger root: a second
# runner refuses to start while it exists. It is removed on a clean exit only. A stale lock (a runner that died
# without exiting) is never cleared automatically: confirm no runner is alive, then delete the runner.lock file
# named in the error by hand to resume.
import atexit
RUNNER_LOCK = os.path.join(SPEND_LEDGER_ROOT, "runner.lock")
try:
    _lock_fd = os.open(RUNNER_LOCK, os.O_CREAT | os.O_EXCL | os.O_WRONLY)
except FileExistsError:
    raise RuntimeError(f"{RUNNER_LOCK} exists. Another runner may be using this run, or a runner stopped without "
                       "releasing it. Refusing to start. Check for a live runner before deleting the lock by hand.")
_RUNNER_TOKEN = f"pid {os.getpid()} {datetime.now(timezone.utc).isoformat()}"
os.write(_lock_fd, _RUNNER_TOKEN.encode())
os.close(_lock_fd)


def _release_runner_lock():
    try:
        with open(RUNNER_LOCK) as f:
            if f.read() == _RUNNER_TOKEN:
                os.remove(RUNNER_LOCK)
    except OSError:
        pass


atexit.register(_release_runner_lock)


print("API key:      loaded")
print(f"Volume:       {VOLUME_MOUNT}  OK")
print(f"Model:        {MODEL_SLUG}")
print(f"Budget:       ${RUN_BUDGET_USD:.2f} per run   ledger: {LEDGER_PATH}")
print(f"Web search:   {USE_WEB_SEARCH}")
print(f"Provider:     {PROVIDER_CONFIG}")
print(f"eCFR:         title {ECFR_TITLE} @ {ECFR_DATE}")
print(f"Mode:         {'SMOKE TEST' if SMOKE_TEST else 'FULL RUN'}")

## eCFR retrieval

Every section number below was verified against the live API. Headings come
from eCFR at runtime, so a wrong label here cannot corrupt a prompt.

In [ ]:
FSVP_SECTIONS = [
    "1.500", "1.501", "1.502", "1.503", "1.504", "1.505", "1.506", "1.507",
    "1.508", "1.509", "1.510", "1.511", "1.512", "1.513", "1.514",
]

RECALL_SECTIONS = [
    "7.40", "7.41", "7.42", "7.45", "7.46", "7.49", "7.50", "7.53", "7.55", "7.59",
    "810.10", "810.11", "810.12", "810.13", "810.15", "810.16", "810.17", "810.18",
]

PCHF_SECTIONS = [
    "117.3", "117.5", "117.126", "117.130", "117.135", "117.139", "117.140",
    "117.150", "117.155", "117.160", "117.165", "117.170", "117.180", "117.190",
]


def fetch_cfr_section(section: str, retries: int = 3):
    """Return (heading, plain_text) for a 21 CFR section, or (None, None)."""
    part = section.split(".")[0]
    url = f"{ECFR_BASE}/{ECFR_DATE}/title-{ECFR_TITLE}.xml"
    for attempt in range(retries):
        try:
            r = requests.get(url, params={"part": part, "section": section},
                             timeout=60, headers=ECFR_HEADERS)
            if r.status_code == 200:
                r.encoding = "utf-8"
                txt = re.sub(r"<[^>]+>", " ", r.text)
                txt = re.sub(r"\s+", " ", txt).strip()
                m = re.match(r"§\s*[\d.]+\s+(.+?\.)\s", txt)
                heading = m.group(1).strip() if m else f"Section {section}"
                return heading, txt
            # v3: 503 was a hard fail in v2, which could drop a whole section.
            # Transient 5xx now retries.
            if r.status_code in (429, 500, 502, 503, 504):
                time.sleep(2 ** (attempt + 2))
                continue
            print(f"  eCFR {section}: HTTP {r.status_code}")
            return None, None
        except Exception as e:
            print(f"  eCFR {section}: {type(e).__name__}")
            time.sleep(2 ** attempt)
    return None, None


def questions_for_length(n_chars: int) -> int:
    """Short sections cannot honestly support 20 distinct questions."""
    if n_chars < 1000:
        return 8
    if n_chars < 2500:
        return 12
    if n_chars < 5000:
        return 16
    return 20


ALL_SECTIONS = (
    [("fsvp", s) for s in FSVP_SECTIONS]
    + [("recall", s) for s in RECALL_SECTIONS]
    + [("pchf", s) for s in PCHF_SECTIONS]
)

print(f"Fetching {len(ALL_SECTIONS)} CFR sections from eCFR...", flush=True)
CFR_TEXT = {}
for _i, (domain, sec) in enumerate(ALL_SECTIONS, 1):
    # v3: v2 printed nothing across 47 network calls, which looked like a hang.
    print(f"  [{_i}/{len(ALL_SECTIONS)}] {sec}", flush=True)
    heading, text = fetch_cfr_section(sec)
    if text:
        CFR_TEXT[sec] = {
            "heading": heading,
            "text": text,
            "chars": len(text),
            "max_questions": questions_for_length(len(text)),
        }
    time.sleep(0.3)

missing = [s for _, s in ALL_SECTIONS if s not in CFR_TEXT]
print(f"\nRetrieved: {len(CFR_TEXT)}/{len(ALL_SECTIONS)}")
if missing:
    print(f"MISSING (will be skipped): {missing}")
print(f"Total CFR text: {sum(v['chars'] for v in CFR_TEXT.values()):,} chars")
print("\nSample headings:")
for s in ["1.506", "810.15", "117.139"]:
    if s in CFR_TEXT:
        v = CFR_TEXT[s]
        print(f"  21 CFR {s:<8} {v['chars']:>6}ch  max_q={v['max_questions']:>2}  {v['heading'][:60]}")

## Seed plan

In [ ]:
ARCHETYPES = [
    "applicability",
    "timeline or deadline",
    "recordkeeping requirement",
    "exemption or qualified exemption",
    "definition",
    "procedural sequence",
    "numeric threshold",
    "responsible party",
    "consequence of non-compliance",
    "conflict between two provisions",
]

BATCH_ALLOCATION = [
    ("fsvp", [s for s in FSVP_SECTIONS if s in CFR_TEXT], 20),
    ("recall", [s for s in RECALL_SECTIONS if s in CFR_TEXT], 20),
    ("pchf", [s for s in PCHF_SECTIONS if s in CFR_TEXT], 10),
]


def build_seeds():
    seeds = []
    for domain, sections, n_batches in BATCH_ALLOCATION:
        if not sections:
            continue
        sec_cycle = itertools.cycle(sections)
        arch_cycle = itertools.cycle(ARCHETYPES)
        seen = set()
        guard = 0
        while len([s for s in seeds if s["domain"] == domain]) < n_batches:
            guard += 1
            if guard > n_batches * 50:
                break
            section, archetype = next(sec_cycle), next(arch_cycle)
            if (section, archetype) in seen:
                continue
            seen.add((section, archetype))
            seeds.append({
                "batch_id": f"{domain}-{len(seen):03d}",
                "domain": domain,
                "section": section,
                "archetype": archetype,
                "n_questions": CFR_TEXT[section]["max_questions"],
            })
    return seeds


SEEDS = build_seeds()
if SMOKE_TEST:
    # One per domain
    picks = []
    for d in ("fsvp", "recall", "pchf"):
        hit = next((s for s in SEEDS if s["domain"] == d), None)
        if hit:
            picks.append(hit)
    SEEDS = picks

print(f"Batches: {len(SEEDS)}")
print(f"By domain: {dict(Counter(s['domain'] for s in SEEDS))}")
print(f"Questions targeted: {sum(s['n_questions'] for s in SEEDS)}")

## Prompt and schema

The section text is in the prompt, so the model is not recalling the CFR —
it is reading it. Web search is for realistic operational scenarios and
current FDA guidance, not for finding the regulation.

In [ ]:
SYSTEM_PROMPT = """You are building an evaluation set for a food safety regulatory model. These questions will score several LLMs on FSQA reasoning; the winner then generates fine-tuning data. Bad questions produce a bad model, so precision matters more than volume.

The full text of the governing CFR section is provided to you. Write questions and answers from that text. Do not rely on memory for what the regulation says.

You have web search. Use it to ground scenarios in real operational detail — actual FDA guidance, warning letters, import refusals, industry practice. Do not use it to look up the regulation; you already have it."""

USER_TEMPLATE = """Generate up to {n} questions.

=== GOVERNING SECTION: 21 CFR {section} — {heading} ===
{section_text}
=== END SECTION ===

ARCHETYPE for this batch: {archetype}

Each question must:
- Be answerable from the section text above
- Be open-ended. No multiple choice, no yes/no
- Describe a real situation a QA manager or FSVP importer would face
- Use the words practitioners use, not regulatory boilerplate
- Be distinct from the others in this batch

Each reference answer must:
- Cite the specific subsection, e.g. "21 CFR {section}(b)(2)"
- Quote or closely track the operative language from the text above
- State the requirement plainly in under 120 words
- Note any condition or exception that changes the answer
- Contain nothing that is not in the section text above

Difficulty mix: roughly 30% straightforward recall, 50% requiring application to the scenario, 20% requiring reconciliation of two or more provisions within the section.

If the section does not support {n} distinct questions of this archetype, return fewer. Do not pad. Every citation must appear in the text above.

Return JSON only, matching the schema."""

RESPONSE_SCHEMA = {
    "type": "json_schema",
    "json_schema": {
        "name": "fsqa_eval_batch",
        "strict": True,
        "schema": {
            "type": "object",
            "properties": {
                "questions": {
                    "type": "array",
                    "items": {
                        "type": "object",
                        "properties": {
                            "question": {"type": "string"},
                            "reference_answer": {"type": "string"},
                            "citation": {"type": "string"},
                            "difficulty": {
                                "type": "string",
                                "enum": ["recall", "application", "reconciliation"],
                            },
                        },
                        "required": ["question", "reference_answer",
                                     "citation", "difficulty"],
                        "additionalProperties": False,
                    },
                }
            },
            "required": ["questions"],
            "additionalProperties": False,
        },
    },
}

CITATION_PATTERN = re.compile(r"21\s*CFR\s*(1\.\d{3}|7\.\d{2}|810\.\d{2}|117\.\d{1,3})")


def norm_citation(q):
    """v3: the model returns 'citation' (str) on some batches and 'citations'
    (list) on others, regardless of what the schema asks for. v2 only read the
    singular key and threw away good questions."""
    c = q.get("citation")
    if isinstance(c, str) and c.strip():
        return c
    cs = q.get("citations")
    if isinstance(cs, list) and cs:
        return "; ".join(str(x) for x in cs)
    return cs if isinstance(cs, str) else ""


print("prompt + schema loaded")

## Probe — verify model, provider pin, and tool combo

Cheap. Run this before anything expensive.

In [ ]:
import threading, uuid
from decimal import Decimal


class BudgetExceeded(RuntimeError):
    pass


PROMPT_OVERHEAD_TOKENS = 256    # allowance for chat-template tokens that the byte bound below does not cover


def max_price_for(model):
    """provider.max_price for one model: its listed price in USD per million tokens (input, output). PRICE already
    holds per-million figures, so the value is converted with Decimal and not scaled. An unpriced model is refused."""
    if model not in PRICE:
        raise BudgetExceeded(f"not sent: no price for {model}. Add it to PRICE in the config cell.")
    pin, pout = PRICE[model]
    return {"prompt": float(Decimal(str(pin))), "completion": float(Decimal(str(pout)))}


def openrouter_body(body):
    """The body every OpenRouter call sends: the caller's body with provider.max_price set from PRICE and usage
    accounting switched on. Done here, once, so no call site can leave either out."""
    if request_enables_search(body):
        require_bounded_search()
    b = dict(body)
    b["provider"] = {**(body.get("provider") or {}), "max_price": max_price_for(body["model"])}
    b["usage"] = {"include": True}
    return b


def worst_case_cost(body):
    """Upper bound for one attempt. Input comes from the UTF-8 byte count of the messages (each token covers
    at least one byte); output is capped by max_tokens, which includes reasoning. Fails closed."""
    model = body["model"]
    if model not in PRICE:
        raise BudgetExceeded(f"not sent: no price for {model}. Add it to PRICE in the config cell.")
    if not isinstance(body.get("max_tokens"), int):
        raise BudgetExceeded(f"not sent: request for {model} has no max_tokens, so its cost cannot be bounded.")
    pin, pout = PRICE[model]
    sent = {k: body[k] for k in ("messages", "tools", "response_format") if k in body}
    in_bound = len(json.dumps(sent, ensure_ascii=False).encode("utf-8")) + PROMPT_OVERHEAD_TOKENS
    return pin / 1e6 * in_bound + pout / 1e6 * body["max_tokens"]


def _is_count(x):
    return isinstance(x, int) and not isinstance(x, bool) and x >= 0


def attempt_cost(body, usage):
    """Settle only against complete, validated provider usage: a reported cost that is finite and above zero,
    and both token counts as non-negative integers. Anything less keeps the worst case that was reserved, so a
    zero, missing, partial or malformed reply cannot release the reservation."""
    if isinstance(usage, dict):
        c = usage.get("cost")
        if (isinstance(c, (int, float)) and not isinstance(c, bool) and 0 < c < float("inf")
                and _is_count(usage.get("prompt_tokens")) and _is_count(usage.get("completion_tokens"))):
            return float(c), "usage"
    return worst_case_cost(body), "worst_case"


_ledger_lock = threading.Lock()
_ledger = {}                    # attempt id -> entry; reloaded from LEDGER_PATH on resume
_budget_halted = False          # set by a refused dispatch or by an overrun; no later attempt is sent
if os.path.exists(LEDGER_PATH):
    with open(LEDGER_PATH) as _f:
        for _line in _f:
            if _line.strip():
                _rec = json.loads(_line)
                _ledger.setdefault(_rec["id"], {}).update(_rec)
    _budget_halted = any(_e.get("overrun") for _e in _ledger.values())   # an overrun halts the run after a restart too


def ledger_total():             # caller holds _ledger_lock. A reserved attempt with no settled line counts at worst case.
    return sum(e["cost"] if e["state"] == "settled" else e["worst_case"] for e in _ledger.values())


def _ledger_write(rec):
    with open(LEDGER_PATH, "a") as f:
        f.write(json.dumps(rec, ensure_ascii=False) + "\n")


def ledger_reserve(body, purpose):
    """Reserve one attempt at its worst case before it is sent. Raises BudgetExceeded if the run is halted, or if
    the reservation would take the run past RUN_BUDGET_USD; landing exactly on the cap is allowed."""
    global _budget_halted
    wc = round(worst_case_cost(body), 6)
    with _ledger_lock:
        if _budget_halted:
            raise BudgetExceeded(f"not sent: the run is halted, so {body['model']} ({purpose}) is refused before dispatch")
        if ledger_total() + wc <= RUN_BUDGET_USD:
            aid = uuid.uuid4().hex
            e = {"id": aid, "state": "reserved", "model": body["model"], "purpose": purpose,
                 "worst_case": wc, "ts": datetime.now(timezone.utc).isoformat()}
            _ledger[aid] = e
            _ledger_write(e)
            return aid
        _budget_halted = True
    raise BudgetExceeded(f"not sent: {body['model']} ({purpose}), worst case ${wc:.4f}, "
                         f"would take the run past RUN_BUDGET_USD = ${RUN_BUDGET_USD:.2f}")


def ledger_settle(aid, body, usage):
    """Settle one attempt. A reported charge above the reservation for this attempt is an overrun: it is written to
    the ledger with its reservation and halts the run, so every later dispatch is refused before any request is sent."""
    global _budget_halted
    cost, source = attempt_cost(body, usage)
    with _ledger_lock:
        e = _ledger[aid]
        charged = round(cost, 6)
        overrun = source == "usage" and charged > e["worst_case"]
        e.update(state="settled", cost=charged, cost_source=source)
        rec = {"id": aid, "state": "settled", "cost": charged, "cost_source": source}
        if overrun:
            e["overrun"] = True
            rec.update(overrun=True, reserved=e["worst_case"])
            _budget_halted = True
        _ledger_write(rec)
    if overrun:
        print(f"Ledger overrun: an attempt was charged ${charged:.4f} against a reservation of ${e['worst_case']:.4f}. "
              f"The run is halted; no later attempt will be sent.")
    return e["cost"]


def ledger_summary():
    with _ledger_lock:
        es, total = list(_ledger.values()), ledger_total()
    by = {}
    for e in es:
        n, c = by.get(e["purpose"], (0, 0.0))
        by[e["purpose"]] = (n + 1, c + (e["cost"] if e["state"] == "settled" else e["worst_case"]))
    lines = [f"Spend ledger: {len(es)} attempts, RUN_BUDGET_USD = ${RUN_BUDGET_USD:.2f}"]
    lines += [f"  {p:8s} attempts={n:5d}  charged=${c:.4f}" for p, (n, c) in sorted(by.items())]
    lines.append(f"  total charged ${total:.4f}   reserved, not settled={sum(e['state'] == 'reserved' for e in es)}")
    return "\n".join(lines)


def guarded_post(body, headers, timeout=REQUEST_TIMEOUT, purpose="call"):
    """One POST to OpenRouter. The attempt is reserved at its worst case before it is sent and settled after, so
    retries, exceptions and rejected calls are all counted. BudgetExceeded is raised before anything is sent."""
    body = openrouter_body(body)
    aid = ledger_reserve(body, purpose)
    usage = None
    try:
        r = requests.post(OPENROUTER_URL, headers=headers, json=body, timeout=timeout)
        if r.status_code == 200:
            try:
                usage = (r.json() or {}).get("usage") or None
            except Exception:
                usage = None
        return r
    finally:
        ledger_settle(aid, body, usage)


def probe(label, body):
    r = guarded_post(body, HEADERS, timeout=180, purpose="probe")
    try:
        d = r.json()
    except Exception:
        print(f"[{label}] HTTP {r.status_code} — unparseable: {r.text[:200]}")
        return False
    if r.status_code == 200:
        prov = d.get("provider")
        byok = d.get("usage", {}).get("is_byok")
        print(f"[OK  ] {label:34} provider={prov} byok={byok}")
        return True
    raw = d.get("error", {}).get("metadata", {}).get("raw", d.get("error", {}).get("message", ""))
    print(f"[FAIL] {label:34} {str(raw)[:130]}")
    return False


_msg = [{"role": "user", "content": "Reply with the single word OK."}]
_base = {"model": MODEL_SLUG, "messages": _msg, "max_tokens": 50,
         "provider": PROVIDER_CONFIG}

print("Probing...\n")
p1 = probe("model + bedrock pin", dict(_base))
p2 = probe("+ web search", {**_base, "tools": WEB_SEARCH_TOOL})
p3 = probe("+ response_format", {**_base, "response_format": RESPONSE_SCHEMA})
p4 = probe("+ both", {**_base, "tools": WEB_SEARCH_TOOL,
                      "response_format": RESPONSE_SCHEMA})

print()
if not p1:
    print("STOP: model or Bedrock pin is not working. Fix before continuing.")
elif not p4 and p2:
    print("NOTE: tools + response_format rejected together.")
    print("      Set USE_RESPONSE_FORMAT = False in cell 4 and re-run that cell.")
elif p4:
    print("All good. Proceed.")

## Generation

In [ ]:
def build_payload(seed, with_response_format=True):
    sec = CFR_TEXT[seed["section"]]
    body = {
        "model": MODEL_SLUG,
        "provider": PROVIDER_CONFIG,
        "max_tokens": GEN_MAX_TOKENS,
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": USER_TEMPLATE.format(
                n=seed["n_questions"],
                section=seed["section"],
                heading=sec["heading"],
                section_text=sec["text"],
                archetype=seed["archetype"],
            )},
        ],
    }
    if USE_WEB_SEARCH:
        require_bounded_search()
        body["tools"] = WEB_SEARCH_TOOL
    if with_response_format and USE_RESPONSE_FORMAT:
        body["response_format"] = RESPONSE_SCHEMA
    return body


def generate_batch(seed: dict) -> dict:
    """One batch. Never raises; returns a result envelope."""
    use_rf = True
    last_error = None

    for attempt in range(MAX_RETRIES):
        try:
            r = guarded_post(build_payload(seed, use_rf), HEADERS, timeout=REQUEST_TIMEOUT, purpose="gen")

            if r.status_code in (429, 503):
                wait = r.headers.get("Retry-After")
                delay = float(wait) if wait and wait.isdigit() else 2 ** (attempt + 2)
                print(f"  [{seed['batch_id']}] {r.status_code} — waiting {delay}s")
                time.sleep(delay)
                last_error = f"HTTP {r.status_code}"
                continue

            if r.status_code >= 400:
                # Capture the body; v1 dropped it.
                last_error = f"HTTP {r.status_code}: {r.text[:400]}"
                print(f"  [{seed['batch_id']}] {last_error}")
                if use_rf and r.status_code == 400:
                    print(f"  [{seed['batch_id']}] retrying without response_format")
                    use_rf = False
                    continue
                time.sleep(2 ** attempt)
                continue

            data = r.json()
            if "error" in data:
                last_error = str(data["error"])[:400]
                print(f"  [{seed['batch_id']}] API error: {last_error}")
                time.sleep(2 ** attempt)
                continue

            content = data["choices"][0]["message"]["content"] or ""
            cleaned = content.replace("```json", "").replace("```", "").strip()
            # Tolerate prose wrapped around the JSON object
            brace = cleaned.find("{")
            if brace > 0:
                cleaned = cleaned[brace:]
            parsed = json.loads(cleaned)
            questions = parsed.get("questions", [])

            sec_text_lower = CFR_TEXT[seed["section"]]["text"].lower()
            clean, rejected = [], []
            for i, q in enumerate(questions):
                q.update({
                    "id": f"{seed['batch_id']}-{i+1:02d}",
                    "batch_id": seed["batch_id"],
                    "domain": seed["domain"],
                    "seed_section": seed["section"],
                    "seed_archetype": seed["archetype"],
                    "section_heading": CFR_TEXT[seed["section"]]["heading"],
                })
                cite = norm_citation(q)
                q["citation"] = cite          # collapse citations[] -> citation
                if not CITATION_PATTERN.search(cite):
                    q["_reject_reason"] = "citation failed format check"
                    rejected.append(q)
                elif seed["section"] not in cite:
                    q["_reject_reason"] = f"citation outside seed section {seed['section']}"
                    rejected.append(q)
                elif len(q.get("question", "")) < 20:
                    q["_reject_reason"] = "question too short"
                    rejected.append(q)
                else:
                    # Grounding signal: does the answer echo language from the section?
                    ans_words = set(re.findall(r"[a-z]{6,}", q.get("reference_answer","").lower()))
                    overlap = sum(1 for w in ans_words if w in sec_text_lower)
                    q["_grounding_overlap"] = round(overlap / max(len(ans_words), 1), 2)
                    clean.append(q)

            print(f"  [{seed['batch_id']}] {len(clean)} kept / {len(rejected)} rejected "
                  f"(asked {seed['n_questions']})")
            return {
                "batch_id": seed["batch_id"], "status": "success",
                "questions": clean, "rejected": rejected,
                "asked": seed["n_questions"], "returned": len(questions),
                "kept": len(clean), "provider": data.get("provider"),
                "usage": data.get("usage", {}), "error": None,
            }

        except (BudgetExceeded, SearchNotBounded):
            raise
        except Exception as e:
            last_error = f"{type(e).__name__}: {str(e)[:400]}"
            print(f"  [{seed['batch_id']}] attempt {attempt+1}: {last_error}")
            time.sleep(2 ** attempt)

    print(f"  [{seed['batch_id']}] FAILED after {MAX_RETRIES} attempts")
    return {
        "batch_id": seed["batch_id"], "status": "failed", "questions": [],
        "rejected": [], "asked": seed["n_questions"], "returned": 0, "kept": 0,
        "provider": None, "usage": {}, "error": last_error,
    }


print("generation function loaded")

In [ ]:
run_id = RUN_ID
print(f"Run {run_id} — {len(SEEDS)} batches, {MAX_WORKERS} workers")
print(f"Web search: {USE_WEB_SEARCH} | response_format: {USE_RESPONSE_FORMAT}")
print("-" * 64)

start = time.time()
with concurrent.futures.ThreadPoolExecutor(max_workers=MAX_WORKERS) as pool:
    results = list(pool.map(generate_batch, SEEDS))
elapsed = time.time() - start

print("-" * 64)
print(f"Done in {elapsed:.0f}s")

## Retry failed batches

v3 addition. A 50-batch run will lose a few to transient errors; retrying the
failures costs seconds and is cheaper than noticing a coverage hole later.

In [ ]:
failed_seeds = [s_ for s_, r_ in zip(SEEDS, results) if r_["status"] != "success"]
if failed_seeds:
    print(f"Retrying {len(failed_seeds)} failed batches: "
          f"{[s_['batch_id'] for s_ in failed_seeds]}", flush=True)
    time.sleep(10)
    with concurrent.futures.ThreadPoolExecutor(max_workers=3) as pool:
        retried = list(pool.map(generate_batch, failed_seeds))
    fixed = 0
    for s_, r_ in zip(failed_seeds, retried):
        if r_["status"] == "success":
            for idx, orig in enumerate(results):
                if orig["batch_id"] == s_["batch_id"]:
                    results[idx] = r_
                    fixed += 1
                    break
    print(f"Recovered {fixed} of {len(failed_seeds)}")
else:
    print("No failed batches to retry.")

## Write to volume

In [ ]:
outdir = os.path.join(VOLUME_MOUNT, OUTPUT_SUBDIR, run_id)
os.makedirs(outdir, exist_ok=True)

all_q, all_rej, failed = [], [], []
tok_in = tok_out = 0
cost = 0.0

for res in results:
    all_q.extend(res["questions"])
    all_rej.extend(res["rejected"])
    if res["status"] != "success":
        failed.append({"batch_id": res["batch_id"], "error": res["error"]})
    u = res.get("usage") or {}
    tok_in += u.get("prompt_tokens", 0)
    tok_out += u.get("completion_tokens", 0)
    cost += u.get("cost", 0) or 0

q_path = os.path.join(outdir, "questions.jsonl")
with open(q_path, "w") as f:
    for q in all_q:
        f.write(json.dumps(q) + "\n")

rej_path = os.path.join(outdir, "rejected.jsonl")
with open(rej_path, "w") as f:
    for q in all_rej:
        f.write(json.dumps(q) + "\n")

# Snapshot the CFR text actually used, so the run is reproducible
cfr_path = os.path.join(outdir, "cfr_sections_used.json")
with open(cfr_path, "w") as f:
    json.dump({k: {"heading": v["heading"], "chars": v["chars"]}
               for k, v in CFR_TEXT.items()}, f, indent=2)

providers_used = sorted({r["provider"] for r in results if r.get("provider")})
grounding = [q["_grounding_overlap"] for q in all_q if "_grounding_overlap" in q]

manifest = {
    "run_id": run_id,
    "generated_at": datetime.now(timezone.utc).isoformat(),
    "smoke_test": SMOKE_TEST,
    "model": MODEL_SLUG,
    "web_search": USE_WEB_SEARCH,
    "response_format": USE_RESPONSE_FORMAT,
    "provider_config": PROVIDER_CONFIG,
    "providers_used": providers_used,
    "ecfr_date": ECFR_DATE,
    "ecfr_sections_retrieved": len(CFR_TEXT),
    "elapsed_seconds": round(elapsed, 1),
    "batches_attempted": len(results),
    "batches_failed": len(failed),
    "failures": failed,
    "questions_kept": len(all_q),
    "questions_rejected": len(all_rej),
    "by_domain": dict(Counter(q["domain"] for q in all_q)),
    "by_difficulty": dict(Counter(q.get("difficulty", "unknown") for q in all_q)),
    "reject_reasons": dict(Counter(q.get("_reject_reason", "?") for q in all_rej)),
    "mean_grounding_overlap": round(sum(grounding) / len(grounding), 3) if grounding else None,
    "yield_by_batch": {r["batch_id"]: {"asked": r["asked"], "kept": r["kept"]}
                       for r in results},
    "underperforming_batches": [r["batch_id"] for r in results
                                if r["asked"] and r["kept"] < r["asked"] * 0.6],
    "tokens": {"prompt": tok_in, "completion": tok_out, "total": tok_in + tok_out},
    "reported_cost_usd": round(cost, 4),
    "files": {"questions": q_path, "rejected": rej_path, "cfr_used": cfr_path},
}

with open(os.path.join(outdir, "manifest.json"), "w") as f:
    json.dump(manifest, f, indent=2)

print(f"Kept:        {manifest['questions_kept']}")
print(f"Rejected:    {manifest['questions_rejected']}  {manifest['reject_reasons']}")
print(f"Failed:      {manifest['batches_failed']} batches")
print(f"By domain:   {manifest['by_domain']}")
print(f"Difficulty:  {manifest['by_difficulty']}")
print(f"Grounding:   {manifest['mean_grounding_overlap']} mean overlap with section text")
print(f"Tokens:      {manifest['tokens']['total']:,}")
print(f"Cost:        ${manifest['reported_cost_usd']}")
print(f"Providers:   {providers_used}")
print(f"Output:      {q_path}")

under = manifest["underperforming_batches"]
if under:
    print(f"\nBatches returning under 60% of asked ({len(under)}): {under[:10]}")
    print("Short CFR sections legitimately support fewer questions; check before assuming a bug.")

if providers_used and providers_used != ["amazon-bedrock"]:
    print("\n*** WARNING ***")
    print("A provider other than amazon-bedrock served part of this run.")
    print("You may be spending OpenRouter credits instead of your own Bedrock key.")

## Smoke test review

The code checks citation *format* and whether the cite falls inside the seed
section. It cannot check that the reference answer is a fair reading of the
regulation. That is what you are doing here.

`grounding` is the fraction of long words in the answer that also appear in
the section text. Low values are worth a closer look — the model may be
writing from memory or from web results rather than the provided text.

In [ ]:
if SMOKE_TEST:
    for q in all_q:
        g = q.get("_grounding_overlap", "?")
        print(f"\n[{q['id']}] {q['difficulty']} | {q['citation']} | grounding={g}")
        print(f"Q: {q['question']}")
        print(f"A: {q['reference_answer']}")

    if all_rej:
        print(f"\n\n{'='*64}\nREJECTED ({len(all_rej)})\n{'='*64}")
        for q in all_rej:
            print(f"\n[{q.get('_reject_reason')}] {q.get('citation','?')}")
            print(f"Q: {q.get('question','')[:160]}")

    print(f"\n{'='*64}")
    print(f"{len(all_q)} kept. Read them. Then set SMOKE_TEST = False and re-run")
    print("cells 6 (seeds) through 13 (write).")

## Utilities

In [ ]:
def list_runs():
    base = os.path.join(VOLUME_MOUNT, OUTPUT_SUBDIR)
    return sorted(os.listdir(base)) if os.path.isdir(base) else []


def load_run(rid: str):
    path = os.path.join(VOLUME_MOUNT, OUTPUT_SUBDIR, rid, "questions.jsonl")
    with open(path) as f:
        return [json.loads(line) for line in f]


def show_section(section: str, n_chars: int = 1500):
    """Print the CFR text the model was given for a section."""
    v = CFR_TEXT.get(section)
    if not v:
        print(f"{section} not retrieved")
        return
    print(f"21 CFR {section} — {v['heading']}  ({v['chars']:,} chars)\n")
    print(v["text"][:n_chars])


print("Available runs:", list_runs())

In [ ]:
import json, os, re

VOLUME_MOUNT = "./local-data"
OUTPUT_SUBDIR = "phase1_fsqa_eval"
# Optional repair pass over one finished run: set RUN to its folder name under OUTPUT_SUBDIR.
# It reads questions.jsonl and rejected.jsonl in that run folder and REWRITES questions.jsonl in place,
# so back up that folder first.
RUN = "<YOUR_RUN_ID>"

CIT = re.compile(r"21\s*CFR\s*\d+\.\d+(?:\([a-z0-9]+\))*")
DIFF_MAP = {"straightforward": "recall", "straightforward recall": "recall",
            "requires application": "application",
            "requires reconciliation": "reconciliation",
            "requires reconciliation of two or more provisions": "reconciliation",
            "recall": "recall", "application": "application",
            "reconciliation": "reconciliation"}

def _checked_repair_run(value):
    """RUN names exactly one existing folder inside VOLUME_MOUNT/OUTPUT_SUBDIR. Refuse a non-string or blank value,
    the unfilled placeholder, an absolute path or drive, any '..' segment, any path separator, a folder whose real
    path is not inside the real path of that root (this covers a symlink that points outside), and anything that is
    not an existing directory. Runs before any file is opened. Fails loudly."""
    if not isinstance(value, str) or not value.strip():
        raise RuntimeError(f"Refusing to repair: RUN must be a non-empty string (got {value!r}).")
    if value.strip() == "<YOUR_RUN_ID>":
        raise RuntimeError("Refusing to repair: RUN is still the placeholder. Set it to the run folder name.")
    if os.path.isabs(value) or value.startswith(("/", "\\")) or os.path.splitdrive(value)[0]:
        raise RuntimeError(f"Refusing to repair: RUN must be a relative name, not an absolute path (got {value!r}).")
    if ".." in value.replace("\\", "/").split("/"):
        raise RuntimeError(f"Refusing to repair: RUN must not contain a '..' segment (got {value!r}).")
    if any(sep in value for sep in ("/", "\\", os.sep, os.altsep) if sep):
        raise RuntimeError(f"Refusing to repair: RUN must be one folder name, with no path separator (got {value!r}).")
    root = os.path.realpath(os.path.join(VOLUME_MOUNT, OUTPUT_SUBDIR))
    real = os.path.realpath(os.path.join(VOLUME_MOUNT, OUTPUT_SUBDIR, value))
    if real == root or os.path.commonpath([root, real]) != root:
        raise RuntimeError(f"Refusing to repair: RUN must resolve to a folder inside {root} (got {value!r}, "
                           f"which resolves to {real}).")
    if not os.path.isdir(real):
        raise RuntimeError(f"Refusing to repair: {os.path.join(VOLUME_MOUNT, OUTPUT_SUBDIR, value)} is not an "
                           "existing folder.")
    return value


RUN = _checked_repair_run(RUN)
d = os.path.join(VOLUME_MOUNT, OUTPUT_SUBDIR, RUN)
kept = [json.loads(l) for l in open(os.path.join(d, "questions.jsonl"))]
rej  = [json.loads(l) for l in open(os.path.join(d, "rejected.jsonl"))]
print(f"before: {len(kept)} kept, {len(rej)} rejected")

for q in rej:
    if q.get("citation", "").strip():
        continue
    m = CIT.search(q.get("reference_answer", ""))
    if m and q["seed_section"] in m.group(0):
        q["citation"] = m.group(0)
        q.pop("_reject_reason", None)
        kept.append(q)

for q in kept:
    q["difficulty"] = DIFF_MAP.get(str(q.get("difficulty", "")).lower().strip(), "application")

seen = set()
kept = [q for q in kept if not (q["id"] in seen or seen.add(q["id"]))]

with open(os.path.join(d, "questions.jsonl"), "w") as f:
    for q in kept:
        f.write(json.dumps(q) + "\n")

from collections import Counter
print(f"after:  {len(kept)} questions")
print(f"  domain     : {dict(Counter(q['domain'] for q in kept))}")
print(f"  difficulty : {dict(Counter(q['difficulty'] for q in kept))}")
print(f"  sections   : {len(set(q['seed_section'] for q in kept))}")